<a href="https://colab.research.google.com/github/andrewgodbout/MCS-3950-F26/blob/main/Resources/A2-tile-kit/tile_kit_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tile kit: make a labelled tile dataset from your project image set

This is how the Assignment 2 dataset was made, done on **your** images (see `README.md` in this
folder for the story and the recipes for Options 4, 8 and 1). Steps:

1. cut tiles from one year of your site(s), with context from the other years
2. label them in a web page (download it, label, export)
3. check how consistent your labels are, and build a train / val / test file split **by site**
4. train a network (your A2 code, or a pretrained one), then map the whole site

Work in your **team repository**, and commit the labels files: they are part of your project.

In [ ]:
import os, sys, glob, urllib.request
import numpy as np
import matplotlib.pyplot as plt
if not os.path.exists("tilekit.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/andrewgodbout/MCS-3950-F26/main/Resources/A2-tile-kit/tilekit.py", "tilekit.py")
import tilekit as tk
IN_COLAB = "google.colab" in sys.modules

## 1. Point at your image set

In Colab, mount Google Drive and set `SITE_DIRS` to your site folder(s): each one holds
`1935.png … 2020.png` and `info.txt`. Use **at least two sites** if you can, so one can be the test
site (Option 4: Charlottetown + your own site).

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

SITE_DIRS = [
    "/content/drive/MyDrive/urban_growth/shared_both_groups/charlottetown",    # <- change
    "/content/drive/MyDrive/urban_growth/group1_stratford/stratford",          # <- change
]
SITE_DIRS = [s for s in SITE_DIRS if os.path.isdir(s)] or sorted(glob.glob("sites/*"))   # demo fallback
for s in SITE_DIRS:
    print(s, tk.site_years(s))

## 2. Cut tiles

* `YEAR`: the year you label. Label each year separately if land use changes (it does).
* `SIZE`, `FACTOR`: the tile is `SIZE` pixels after averaging `FACTOR` × `FACTOR`. At 1 m,
  `SIZE=64, FACTOR=1` is 64 m; A2 used about 148 m.
* `STRATEGY="texture"` spreads tiles from smooth to busy areas, so rare things are not buried.

In [ ]:
YEAR, SIZE, FACTOR, N_PER_SITE = 1958, 64, 2, 150
CLASSES = ["urban", "agriculture", "vegetated"]          # <- your classes, written down as rules first
OUT = f"tiles_{YEAR}"

index = tk.cut_tiles(SITE_DIRS, YEAR, size=SIZE, factor=FACTOR, n=N_PER_SITE,
                     strategy="texture", out_dir=OUT)
X = np.load(f"{OUT}/tiles.npz")["X"]
fig, axes = plt.subplots(3, 10, figsize=(14, 4.6))
for ax, x in zip(axes.ravel(), X[np.random.default_rng(0).choice(len(X), 30, replace=False)]):
    ax.imshow(x, cmap="gray", vmin=0, vmax=255); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Label

This writes `labeller.html`. **Download it and open it in your browser** (double-click). Hover a
tile and press a number key, `0` to skip. When you're done, or need a break, click **Export labels**:
it downloads `labels_<your name>.json`. Put that file in `tiles_<year>/` (upload it below in Colab)
and commit it.

Split the pages between teammates, and have two people label the same 50 tiles so you can
measure agreement.

In [ ]:
page = tk.make_labeller(OUT, CLASSES, title=f"{YEAR} tiles")
if IN_COLAB:
    from google.colab import files
    files.download(str(page))

In [ ]:
# Colab: upload your exported labels_*.json files into the tiles folder
if IN_COLAB:
    from google.colab import files
    for name, data in files.upload().items():
        open(os.path.join(OUT, name), "wb").write(data)
print(sorted(glob.glob(f"{OUT}/labels_*.json")))

## 4. Check your labels and build the dataset

In [ ]:
LABELS = glob.glob(f"{OUT}/labels_*.json")
tk.agreement(LABELS)
TEST_SITES = [os.path.basename(SITE_DIRS[-1])] if len(SITE_DIRS) > 1 else []   # held-out place
tk.build_dataset(OUT, LABELS, CLASSES, test_sites=TEST_SITES, out=f"dataset_{YEAR}.npz")

The file has the same layout as `A2_tiles_1958.npz` (`X_train`, `y_train`, …, `classes`, `mean`,
`std`), so your Assignment 2 code runs on it with one change: the number of outputs in the last
layer. With only a few hundred tiles, also try L14's frozen pretrained features.

## 5. Map the whole site

Wrap your trained model in a `predict` function (uint8 tiles in, class probabilities out) and
slide it over a whole year. Below, a stand-in "model" (darkness and texture thresholds) shows
the mechanics; replace it with yours.

In [ ]:
def predict(tiles):                     # <- replace with your trained network, e.g.
    # x = ((torch.as_tensor(tiles, dtype=torch.float32) - MEAN) / STD).unsqueeze(1)
    # model.eval();  with torch.no_grad(): return torch.softmax(model(x), 1).numpy()
    t = tiles.astype(float)
    tex = np.abs(np.diff(t, axis=2)).mean((1, 2)); dark = t.mean((1, 2)) < 110
    cls = np.where(tex > 18, 2, np.where(dark & (tex > 10), 0, 1))
    return np.eye(len(CLASSES))[cls]

img, nodata = tk.load_year(SITE_DIRS[0], YEAR)
M, P = tk.sliding_map(predict, img, SIZE, stride=SIZE // 2, factor=FACTOR, nodata=nodata)
fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
ax[0].imshow(img, cmap="gray"); ax[0].set_title(f"{os.path.basename(SITE_DIRS[0])} {YEAR}")
im = ax[1].imshow(np.ma.masked_less(M, 0), cmap="tab10", vmin=0, vmax=9, interpolation="nearest")
ax[1].set_title("predicted class per cell")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
print({c: f"{(M == i).mean():.0%}" for i, c in enumerate(CLASSES)})

Do this for every year and you have a change map; the share of each class per year is a
first answer to your project question. Before you believe it, check a few cells by eye, and run
an occlusion map (A2 Part 6) on some tiles.